In [4]:
MODEL_ID='CohereLabs/tiny-aya-global'
import os
from dotenv import load_dotenv
load_dotenv("keys.env")
assert os.environ["HF_TOKEN"].startswith("hf"),\
   "hugging face login failed"

In [5]:
from transformers import AutoTokenizer, pipeline

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

messages = [
    {"role": "system", "content": "You are a friendly chatbot who always responds in the style of a pirate",},
    {"role": "user", "content": "How many helicopters can a human eat in one sitting?"},
 ]
tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True)
pipe = pipeline(
    task="text-generation", 
    model=MODEL_ID,
    use_fast=True,
    kwargs={
        "return_full_text": False,
    },
    model_kwargs={},
    tokenizer=tokenizer
)


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

In [6]:
print(tokenizer.chat_template)

{'default': '{{ bos_token }}{% set ns = namespace(system_prompt=false, expect_user=true) %}{% for message in messages %}{% if message[\'role\']|lower == \'system\' %}{% set ns.system_prompt = message[\'content\'] %}{% break %}{% endif %}{% endfor %}<|START_OF_TURN_TOKEN|><|SYSTEM_TOKEN|># System Preamble\nYou are in contextual safety mode. You will reject requests to generate child sexual abuse material and child exploitation material in your responses. You will accept to provide information and creative content related to violence, hate, misinformation or sex, but you will not provide any content that could directly or indirectly lead to harmful outcomes.\n\nYour information cutoff date is June 2024.\n\nYou have been trained on data in English, Dutch, French, Italian, Portuguese, Romanian, Spanish, Czech, Polish, Ukrainian, Russian, Greek, German, Danish, Swedish, Norwegian, Catalan, Galician, Welsh, Irish, Basque, Croatian, Latvian, Lithuanian, Slovak, Slovenian, Estonian, Finnish, H

In [7]:
def generate_product_description(item: str):
  system_prompt = """You are a product marketer for a company that makes nutrition
  supplements. Balance your product descriptions to attract customers, optimize
SEO, and stay within accurate advertising guidelines.
"""
  user_prompt = f"""Write a product description for a {item}."""
  input_message = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": user_prompt}
]

  results = pipe(input_message, max_new_tokens=512)
  return results[0]['generated_text'][-1]['content'].strip()
prod = generate_product_description("protein drink")
print(prod)

Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


**Boost Your Energy with our Premium Protein Drink – Fuel Your Fitness Journey**  

Stay ahead with **Protein Power**, the ultimate dietary supplement designed to fuel your workouts and support your health goals. Whether you're a fitness enthusiast, an athlete, or simply looking to maintain a balanced lifestyle, our protein drink offers a powerful blend of essential amino acids, complete proteins, and essential nutrients to keep you energized and performing at your best.  

**Why Choose Protein Power?**  
- **High-Quality Protein**: Each serving contains [X] grams of complete protein, derived from premium sources like whey, plant-based proteins, and collagen, ensuring optimal muscle recovery and growth.  
- **Natural Flavors & Ingredients**: Enjoy a range of delicious flavors like Vanilla Bean, Chocolate Fudge, and Berry Blast, made with natural sweeteners instead of artificial additives.  
- **Quick Absorption & Recovery**: Whether you're pre-training, post-workout, or just need a pic

In [7]:
with open("banned_phrases.txt") as ifp:
    banned_phrases = [line.strip().lower() for  line in ifp.readlines()]
banned_phrases[100:105]

['decomposable', 'definitive', 'degradable', 'dementia', 'depression']

In [8]:
with open("desired_phrases.txt") as ifp:
    desired_phrases = [line.strip().lower() for  line in ifp.readlines()]
desired_phrases[:5]

['nutrition', 'calorie deficit', 'diet', 'protein shake', 'paleo diet']

In [9]:
banned_phrases = set(banned_phrases)
desired_phrases = set(desired_phrases)
type(desired_phrases)

set

In [ ]:
import numpy as np

def evaluate(descr: str, positives: set, negatives: set) -> int:
    descr = descr.lower()
    num_positive = np.sum([1 for phrase in positives if phrase in descr])
    num_negative = np.sum([1 for phrase in negatives if phrase in descr])
    return int(num_positive - num_negative)

def evaluate_verbose(descr: str, positives: set, negatives: set) -> int:
    descr = descr.lower()
    num_positive = num_negative = 0
    for phrase in positives:
        if phrase in descr:
            num_positive += 1
            print(f"Good: {phrase}")
    for phrase in negatives:
        if phrase in descr:
            num_negative += 1
            print(f"Bad: {phrase}")
    print(f"Good: {num_positive}, Bad: {num_negative}")
    return num_positive - num_negative

evaluate_verbose(prod, desired_phrases, banned_phrases)
        

In [19]:
evaluate(prod, desired_phrases, banned_phrases)

-8

In [1]:
import torch
import numpy as np
from transformers.generation.logits_process import (
    LogitsProcessor,
    LOGITS_PROCESSOR_INPUTS_DOCSTRING
)
from transformers.utils import add_start_docstrings

class BrandLogitsProcessor(LogitsProcessor):
    def __init__(self, tokenizer, positives, negatives):
        self.tokenizer = tokenizer
        self.positives = positives
        self.negatives = negatives
    
    @add_start_docstrings(LOGITS_PROCESSOR_INPUTS_DOCSTRING)
    def __call__(
        self,input_ids: torch.LongTensor, input_logits: torch.FloatTensor
    ) -> torch.FloatTensor:
        output_logits = input_logits.clone()

        num_matches = [0] * len(input_ids)
        for idx, seq in enumerate(input_ids):
            decoded = self.tokenizer.decode(seq)
            num_matches[idx] = evaluate(decoded, self.positives, self.negatives)
        max_matches = np.max(num_matches)
        for idx in range (len(input_ids)):
            if num_matches[idx] != max_matches:
                output_logits[idx] = -10000
        
        return output_logits

In [13]:
def generate_product_description_v2(item: str) -> str:
    system_prompt = """
        You are a product marketer for a company that makes nutrition supplements.
        Balance your product descriptions to attract customers, optimize SEO, and
        stay within accurate advertising guidelines.
        Product descriptions have to be 3-5 sentences.
        Provide only the product description with no preamble.
    """
    user_prompt = f"Write a product description for {item}"
    
    input_message = [
        {"role": "system", "content" : system_prompt},
        {"role": "user", "content" : user_prompt}
    ]
    brand_processor = BrandLogitsProcessor(pipe.tokenizer, desired_phrases, banned_phrases)
    result = pipe(input_message,
                  max_new_tokens=512,
                  do_sample=True,
                  temperature=0.8,
                  num_beams=10,
                  use_cache=True,
                  logits_processor=[brand_processor])
    return result[0]['generated_text'][-1]['content'].strip()
prod = generate_product_description_v2("protein drink")
print(prod)

Both `max_new_tokens` (=512) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Elevate your fitness journey with our premium protein drink, designed to fuel your workouts and support muscle recovery. Packed with 25g of clean, fast-absorbing protein per serving, it provides the essential amino acids your body needs to repair and build lean muscle tissue. Whether you're a busy professional, an athlete, or a fitness enthusiast, this protein shake delivers essential nutrients in a convenient, low-calorie format. Mix it with water, milk, or your favorite beverage for a quick and delicious way to meet your daily protein goals. Experience the difference in your strength, endurance, and overall well-being with every sip.


In [15]:
evaluate(prod, desired_phrases, banned_phrases)
evaluate_verbose(prod, desired_phrases, banned_phrases)

Good: calorie
Good: nutrients
Good: protein shake
Good: 3, Bad: 0


3